
# PyBlastAfterglowMag：Gaussian Jet 数据模拟（1000组）- 带重新采样机制

这个 notebook 按照你原来的 VegasAfterglow notebook 逻辑改写，并添加了**重新采样机制**：

- 先定义参数空间
- 用 LHS 采样 1000 组物理参数
- 逐组构造 PyBlastAfterglowMag 的 Gaussian jet + ISM 配置
- 运行前向模拟并提取 light curve
- **如果遇到错误（如频率超出范围、数值不稳定等），自动重新采样该参数组**
- 保存成长表数据集，方便后续做 surrogate / transfer learning

## 主要改进

1. **重新采样机制**：当某组参数导致模拟失败时，自动重新采样该参数组（最多重试5次）
2. **频率范围修正**：调整 NU_MAX 以适应 PyBlastAfterglowMag 内部计算范围
3. **更严格的参数约束**：避免产生数值不稳定的参数组合
4. **详细的错误日志**：记录每次失败的原因和重试次数


In [39]:

import os
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from dataclasses import dataclass
from enum import Enum
from tqdm.auto import tqdm

from PyBlastAfterglowMag.wrappers import run_grb
from PyBlastAfterglowMag.utils import cgs


## 一、全局配置

In [40]:

# ====== 数据规模 ======
N_THETA = 50   # 物理参数组数
K_TIME  = 64     # 每组参数保留的时间点数
M_FREQ  = 4      # 每组参数保留的频率点数

# ====== 全局时间范围（秒）=====
T_MIN = 3e3
T_MAX = 1e7

# ====== 全局频率范围（Hz）=====
# 注意：PyBlastAfterglowMag 内部计算的频率范围有限
# 根据错误信息，内部最大频率约为 3e13 Hz，所以我们设置 NU_MAX = 1e12
#NU_MIN = 1e9
#NU_MAX = 1e12    # 修改为 1e12，避免超出内部计算范围

# ====== 红移范围 ======
Z_MIN = 0.01
Z_MAX = 1.0      # 限制红移范围，避免数值问题

# ====== 结构喷流附加参数范围 ======
THETA_W_MIN = np.deg2rad(5.0)
THETA_W_MAX = np.deg2rad(30.0)

# ====== 重新采样设置 ======
MAX_RETRIES = 5  # 每组参数最大重试次数

# ====== 数值设置 ======
RANDOM_SEED = 20260308
FLOOR_FLUX = 1e-300
RTOL = 5e-7
TB0, TB1, NTB = 3e2, 1e12, 600
N_LAYERS_A = 21

# ====== 路径设置 ======
WORK_ROOT = Path('./pyblast_runs_gaussian_1000')
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# !!! 必改：改成你本机的 pba.out 路径 !!!
PATH_TO_CPP = '/mnt/data/s2119250005/PyBlastAfterglowMag/src/pba.out'

# 输出文件名
OUT_PARQUET = 'pyblast_gaussian_point_dataset_1000.parquet'
OUT_CSV = 'pyblast_gaussian_point_dataset_1000.csv'
OUT_THETA_CSV = 'pyblast_gaussian_theta_table_1000.csv'

rng = np.random.default_rng(RANDOM_SEED)


## 二、定义参数（log/linear 混合）+ LHS 采样工具

In [41]:

class Scale(Enum):
    LINEAR = 'linear'
    LOG = 'log'

@dataclass(frozen=True)
class ParamSpec:
    name: str
    low: float
    high: float
    scale: Scale

PARAM_SPECS = [
    ParamSpec('z',       0.01,   1.0,    Scale.LINEAR),
    ParamSpec('theta_v', 0.0,    0.35,   Scale.LINEAR),
    ParamSpec('Gamma0',  80.0,   400.0,  Scale.LOG),
    ParamSpec('E_iso',   1e51,   3e54,   Scale.LOG),
    ParamSpec('theta_c', 0.03,   0.20,   Scale.LOG),
    ParamSpec('theta_w', np.deg2rad(10.0), np.deg2rad(25.0), Scale.LINEAR),
    ParamSpec('n_ism',   1e-3,   1e-1,   Scale.LOG),
    ParamSpec('p',       2.05,   2.4,    Scale.LINEAR),
    ParamSpec('eps_e',   1e-4,   1e-1,   Scale.LOG),
    ParamSpec('eps_B',   1e-6,   1e-3,   Scale.LOG),
]

def sample_lhs_unit(n: int, d: int, seed: int) -> np.ndarray:
    try:
        from scipy.stats import qmc
        sampler = qmc.LatinHypercube(d=d, seed=seed)
        return sampler.random(n=n)
    except ImportError:
        print('WARNING: scipy 未安装，LHS 将退化为普通 uniform 采样。建议安装 scipy。')
        r = np.random.default_rng(seed)
        return r.random((n, d))

def transform_unit_to_param(u: np.ndarray, spec: ParamSpec) -> np.ndarray:
    if spec.scale == Scale.LINEAR:
        return spec.low + u * (spec.high - spec.low)
    if spec.scale == Scale.LOG:
        lo, hi = np.log10(spec.low), np.log10(spec.high)
        return 10 ** (lo + u * (hi - lo))
    raise ValueError(f'Unknown scale: {spec.scale}')


## 三、物理约束 + 采样参数表（1000组）

In [42]:
def apply_physical_constraints(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # core angle 限制在更稳定区间
    df['theta_c'] = np.clip(df['theta_c'].values, 0.03, 0.20)

    # wing angle 必须明显大于 core
    df['theta_w'] = np.maximum(df['theta_w'].values, 2.5 * df['theta_c'].values)
    df['theta_w'] = np.clip(df['theta_w'].values, 0.18, 0.45)

    # observer angle 留出边界裕量，不要等于 theta_w
    df['theta_v'] = np.minimum(df['theta_v'].values, 0.8 * df['theta_w'].values)

    # 再限制一个最大离轴比例，避免 theta_v/theta_c 过大
    df['theta_v'] = np.minimum(df['theta_v'].values, 6.0 * df['theta_c'].values)
    
    df['n_ism'] = np.clip(df['n_ism'].values, 1e-3, 1e-1)  # 最低 1e-3 cm^-3
    return df

def sample_single_params(specs: list[ParamSpec], rng: np.random.Generator) -> dict:
    """采样单组参数"""
    u = rng.random(len(specs))
    data = {}
    for j, spec in enumerate(specs):
        data[spec.name] = transform_unit_to_param(u[j], spec)
    return data

def sample_parameter_table(n_theta: int, specs: list[ParamSpec], seed: int) -> pd.DataFrame:
    d = len(specs)
    u = sample_lhs_unit(n=n_theta, d=d, seed=seed)

    data = {}
    for j, spec in enumerate(specs):
        data[spec.name] = transform_unit_to_param(u[:, j], spec)

    df = pd.DataFrame(data)
    df = apply_physical_constraints(df)
    df.insert(0, 'theta_id', np.arange(len(df), dtype=int))
    return df

# 初始采样
theta_df = sample_parameter_table(N_THETA, PARAM_SPECS, seed=RANDOM_SEED)
theta_df.head()


,theta_id,z,theta_v,Gamma0,E_iso,theta_c,theta_w,n_ism,p,eps_e,eps_B
0,0,0.133159,0.215264,282.178589,2.486071e+54,0.070959,0.410033,0.042564,2.167659,0.048042,0.000008
1,1,0.197189,0.163089,212.160475,7.329994e+52,0.040110,0.203861,0.061424,2.264027,0.001988,0.000002
2,2,0.752529,0.158377,201.693562,3.054017e+51,0.046099,0.285415,0.002828,2.318408,0.000181,0.000001
3,3,0.646919,0.140224,209.270577,8.573694e+51,0.099838,0.249594,0.009700,2.145060,0.020512,0.000012
4,4,0.604972,0.215473,178.082136,1.739555e+51,0.035912,0.338195,0.002532,2.390015,0.009257,0.000003


## 四、z → 光度距离 d_L（cm）

In [43]:

def luminosity_distance_cm(z: float) -> float:
    try:
        from astropy.cosmology import Planck18 as cosmo
        return cosmo.luminosity_distance(z).to('cm').value
    except Exception:
        pass

    try:
        from scipy.integrate import quad
    except ImportError as e:
        raise ImportError(
            '需要 astropy 或 scipy 才能从 z 计算光度距离。建议：pip install astropy 或 pip install scipy'
        ) from e

    c_km_s = 299792.458
    H0 = 67.66
    Om0 = 0.3111
    Ol0 = 1.0 - Om0
    Mpc_to_cm = 3.085677581e24

    def E(zp):
        return np.sqrt(Om0 * (1 + zp) ** 3 + Ol0)

    integral, _ = quad(lambda zp: 1.0 / E(zp), 0, z, limit=200)
    d_c_mpc = (c_km_s / H0) * integral
    d_l_mpc = (1 + z) * d_c_mpc
    return d_l_mpc * Mpc_to_cm


## 五、PyBlast 配置构造 + 单样本模拟（带错误处理）

In [44]:

def build_pba_config_from_row(row: pd.Series,
                              k_time: int,
                              m_freq: int,
                              t_min: float,
                              t_max: float,
                              nu_min: float,
                              nu_max: float) -> dict:
    z = float(row['z'])
    d_l = luminosity_distance_cm(z)

    # 定义4个固定频率（单位：Hz），必须低于 1e11 Hz
    fixed_freqs = [1e9, 5e9, 1e10, 5e10]  # 1GHz, 5GHz, 10GHz, 50GHz

    main_pars = dict(
        d_l=d_l,
        z=z,
        n_ism=float(row['n_ism']),
        theta_obs=float(row['theta_v']),
        rtol=RTOL,
         # 【修改这里】从 logspace 改为固定值列表
        lc_freqs=f'array {" ".join([str(f) for f in fixed_freqs])}',
        lc_times=f'array logspace {t_min} {t_max} {k_time}',
        tb0=TB0,
        tb1=TB1,
        ntb=NTB,
    )

    struct = dict(
        struct='gaussian',
        Eiso_c=float(row['E_iso']),
        Gamma0c=float(row['Gamma0']),
        M0c=-1.0,
        theta_c=float(row['theta_c']),
        theta_w=float(row['theta_w']),
        n_layers_a=N_LAYERS_A,
    )

    grb_pars = dict(
        structure=struct,
        eps_e_fs=float(row['eps_e']),
        eps_b_fs=float(row['eps_B']),
        p_fs=float(row['p']),
        do_lc='yes',
        save_spec='no',
        method_synchrotron_fs='CSYN',
        method_ele_fs='numeric',
    )

    return {'main': main_pars, 'grb': grb_pars}


def run_single_simulation(row: pd.Series,
                          work_root: Path,
                          path_to_cpp: str,
                          k_time: int,
                          m_freq: int,
                          t_min: float,
                          t_max: float,
                          nu_min: float,
                          nu_max: float,
                          overwrite: bool = True) -> pd.DataFrame:
    """
    运行单组参数的模拟
    如果失败，返回 None
    """
    theta_id = int(row['theta_id'])
    working_dir = work_root / f'run_{theta_id:04d}'

    if overwrite and working_dir.exists():
        shutil.rmtree(working_dir)
    working_dir.mkdir(parents=True, exist_ok=True)

    P = build_pba_config_from_row(
        row=row,
        k_time=k_time,
        m_freq=m_freq,
        t_min=t_min,
        t_max=t_max,
        nu_min=nu_min,
        nu_max=nu_max,
    )

    try:
        pba = run_grb(
            working_dir=str(working_dir) + '/',
            P=P,
            run=True,
            path_to_cpp=path_to_cpp,
            loglevel='err',
            process_skymaps=False,
        )

        ejecta = pba.GRB
        times = np.asarray(ejecta.get_lc_times(), dtype=float)
        # 【修改这里】使用固定频率列表，不再从 nu_min/nu_max 生成
        freqs = [1e9, 5e9, 1e10, 5e10]  # 必须与上面配置完全一致

        chunks = []
        for freq in freqs:
            lc = np.asarray(ejecta.get_lc(freq=float(freq)), dtype=float)
            df_i = pd.DataFrame({
                'theta_id': theta_id,
                't_s': times,
                'nu_hz': float(freq),
                'F_nu_mJy': lc,
            })
            for col in row.index:
                if col != 'theta_id':
                    df_i[col] = row[col]
            chunks.append(df_i)

        out = pd.concat(chunks, ignore_index=True)
        out['log10_t'] = np.log10(out['t_s'])
        out['log10_nu'] = np.log10(out['nu_hz'])

        finite = np.isfinite(out['F_nu_mJy'].values)
        positive = out['F_nu_mJy'].values > 0
        out['valid'] = finite & positive

        safe_flux = np.maximum(out['F_nu_mJy'].values, FLOOR_FLUX)
        out['log10_F_nu_mJy'] = np.where(out['valid'].values, np.log10(safe_flux), np.nan)
        
        return out
        
    except Exception as e:
        error_msg = str(e)
        # 清理失败的目录
        if working_dir.exists():
            shutil.rmtree(working_dir)
        raise RuntimeError(error_msg)


## 六、批量生成点样本数据集（带重新采样）

In [45]:

def generate_point_dataset_with_resampling(
                           theta_df: pd.DataFrame,
                           work_root: Path,
                           path_to_cpp: str,
                           k_time: int,
                           m_freq: int,
                           t_min: float,
                           t_max: float,
                           nu_min: float,
                           nu_max: float,
                           max_retries: int = 5,
                           overwrite_each_run: bool = True) -> pd.DataFrame:
    """
    批量生成数据集，带重新采样机制
    如果某组参数失败，会重新采样该参数组并重试
    """
    chunks = []
    failures = []
    resample_log = []  # 记录重新采样信息
    
    # 用于重新采样的随机数生成器
    resample_rng = np.random.default_rng(RANDOM_SEED + 99999)

    for idx, row in tqdm(theta_df.iterrows(), total=len(theta_df)):
        theta_id = int(row['theta_id'])
        success = False
        retry_count = 0
        current_row = row.copy()
        
        while not success and retry_count <= max_retries:
            try:
                df_i = run_single_simulation(
                    row=current_row,
                    work_root=work_root,
                    path_to_cpp=path_to_cpp,
                    k_time=k_time,
                    m_freq=m_freq,
                    t_min=t_min,
                    t_max=t_max,
                    nu_min=nu_min,
                    nu_max=nu_max,
                    overwrite=overwrite_each_run,
                )
                chunks.append(df_i)
                success = True
                
                # 记录重新采样信息
                if retry_count > 0:
                    resample_log.append({
                        'theta_id': theta_id,
                        'retries': retry_count,
                        'final_success': True
                    })
                    print(f"  [RESAMPLED] theta_id={theta_id} 重试{retry_count}次后成功")
                    
            except Exception as e:
                retry_count += 1
                error_msg = str(e)
                
                if retry_count > max_retries:
                    # 超过最大重试次数，记录失败
                    print(f"[FAILED] theta_id={theta_id} 超过最大重试次数({max_retries})")
                    print(f"  最后错误: {error_msg[:100]}...")
                    failures.append({
                        'theta_id': theta_id,
                        'error': error_msg,
                        'retries': retry_count - 1,
                        'final_params': current_row.to_dict()
                    })
                    resample_log.append({
                        'theta_id': theta_id,
                        'retries': retry_count - 1,
                        'final_success': False,
                        'error': error_msg
                    })
                else:
                    # 重新采样该参数组
                    print(f"[RETRY] theta_id={theta_id} 第{retry_count}次重试，错误: {error_msg[:80]}...")
                    
                    # 重新采样参数
                    new_params = sample_single_params(PARAM_SPECS, resample_rng)
                    
                    # 应用物理约束
                    new_df = pd.DataFrame([new_params])
                    new_df = apply_physical_constraints(new_df)
                    
                    # 更新 current_row，但保留 theta_id
                    current_row = new_df.iloc[0]
                    current_row['theta_id'] = theta_id
                    
                    # 更新 theta_df 中的对应行（用于最终保存）
                    for col in new_params.keys():
                        theta_df.at[idx, col] = current_row[col]

    if len(chunks) == 0:
        raise RuntimeError('所有样本都运行失败，请检查 PATH_TO_CPP、PyBlast 安装和参数设置。')

    point_df = pd.concat(chunks, ignore_index=True)
    fail_df = pd.DataFrame(failures)
    resample_df = pd.DataFrame(resample_log)
    
    print(f"\n=== 生成完成 ===")
    print(f"成功样本数: {len(chunks)} / {len(theta_df)}")
    print(f"失败样本数: {len(failures)}")
    if len(resample_log) > 0:
        resampled_count = sum(1 for x in resample_log if x.get('final_success', False))
        print(f"重新采样后成功: {resampled_count}")
    
    return point_df, fail_df, resample_df


## 七、正式生成 1000 组数据（带重新采样）

In [46]:

# 运行批量生成，带重新采样机制
point_df, fail_df, resample_df = generate_point_dataset_with_resampling(
    theta_df=theta_df,
    work_root=WORK_ROOT,
    path_to_cpp=PATH_TO_CPP,
    k_time=K_TIME,
    m_freq=M_FREQ,
    t_min=T_MIN,
    t_max=T_MAX,
    nu_min=NU_MIN,
    nu_max=NU_MAX,
    max_retries=MAX_RETRIES,
    overwrite_each_run=True,
)

print('\npoint_df.shape =', point_df.shape)
print('fail_df.shape  =', fail_df.shape if len(fail_df) > 0 else '(empty)')
print('resample_df.shape =', resample_df.shape if len(resample_df) > 0 else '(empty)')
point_df.head()


  2%|▏         | 1/50 [10:28<8:33:25, 628.67s/it]

 mom=279.528 Gamma=279.53 beta=0.999994
 mom=259.218 Gamma=259.219 beta=0.999993
 mom=222.931 Gamma=222.934 beta=0.99999
 mom=177.835 Gamma=177.838 beta=0.999984
 mom=131.626 Gamma=131.63 beta=0.999971
 mom=90.4544 Gamma=90.4599 beta=0.999939
 mom=57.7891 Gamma=57.7977 beta=0.99985
 mom=34.4166 Gamma=34.4311 beta=0.999578
 mom=19.2166 Gamma=19.2426 beta=0.998649
 mom=10.1797 Gamma=10.2287 beta=0.99521
 mom=5.23357 Gamma=5.32825 beta=0.98223
 mom=2.70286 Gamma=2.88192 beta=0.937868
 mom=1.44659 Gamma=1.75859 beta=0.822588
 mom=0.804569 Gamma=1.28348 beta=0.626863
 mom=0.453951 Gamma=1.09821 beta=0.413354
 mom=0.253148 Gamma=1.03154 beta=0.245407
 mom=0.137382 Gamma=1.00939 beta=0.136104
 mom=0.0720594 Gamma=1.00259 beta=0.071873
 mom=0.0364364 Gamma=1.00066 beta=0.0364122
 mom=0.0177457 Gamma=1.00016 beta=0.0177429
 mom=0.00832243 Gamma=1.00003 beta=0.00832215
 mom=210.618 Gamma=210.62 beta=0.999989
 mom=198.691 Gamma=198.694 beta=0.999987
 mom=176.836 Gamma=176.839 beta=0.999984
 mom=1

[ ERROR   ] : [ blastwave.h:1527 ] : /tmp/ipykernel_1985683/1492004778.py:89: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current_row['theta_id'] = theta_id
/tmp/ipykernel_1985683/1492004778.py:89: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current_row['theta_id'] = theta_id
/mnt/data/s2119250005/anaconda3/envs/GRB/lib/python3.9/site-packages/PyBlastAfterglowMag/id_analytic.py:131: RuntimeWarning: divide by zero encountered in scalar divide
  self.dist_M0_a[i] = self.dist_E0_a[i] / (( Gamma - 1.0) * c*c )
  4%|▍         | 2/50 [20:25<8:07:50, 609.80s/it]

 mom=280.953 Gamma=280.954 beta=0.999994
 mom=110.235 Gamma=110.239 beta=0.999959
 mom=17.6042 Gamma=17.6326 beta=0.998391
 mom=1.71838 Gamma=1.98817 beta=0.864301
 mom=0.215271 Gamma=1.02291 beta=0.21045
 mom=0.0203593 Gamma=1.00021 beta=0.0203551
 mom=0.00120952 Gamma=1 beta=0.00120952
 mom=4.48881e-05 Gamma=1 beta=4.48881e-05
  [RESAMPLED] theta_id=1 重试1次后成功
 mom=199.522 Gamma=199.525 beta=0.999987
 mom=182.994 Gamma=182.997 beta=0.999985
 mom=153.952 Gamma=153.955 beta=0.999979
 mom=118.841 Gamma=118.845 beta=0.999965
 mom=84.2297 Gamma=84.2356 beta=0.99993
 mom=54.8869 Gamma=54.896 beta=0.999834
 mom=32.9777 Gamma=32.9929 beta=0.999541
 mom=18.3829 Gamma=18.41 beta=0.998524
 mom=9.63375 Gamma=9.68551 beta=0.994656
 mom=4.87069 Gamma=4.97229 beta=0.979568
 mom=2.47077 Gamma=2.66546 beta=0.926956
 mom=1.30003 Gamma=1.64015 beta=0.792631
 mom=0.708528 Gamma=1.22557 beta=0.578123
 mom=0.388636 Gamma=1.07286 beta=0.362242
 mom=0.208857 Gamma=1.02158 beta=0.204446
 mom=0.108399 Gamma=1.

[ ERROR   ] : [ blastwave.h:1527 ] : /tmp/ipykernel_1985683/1492004778.py:89: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current_row['theta_id'] = theta_id
/tmp/ipykernel_1985683/1492004778.py:89: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current_row['theta_id'] = theta_id
/mnt/data/s2119250005/anaconda3/envs/GRB/lib/python3.9/site-packages/PyBlastAfterglowMag/id_analytic.py:131: RuntimeWarning: divide by zero encountered in scalar divide
  self.dist_M0_a[i] = self.dist_E0_a[i] / (( Gamma - 1.0) * c*c )
  6%|▌         | 3/50 [31:03<8:07:52, 622.83s/it]

...

 98%|█████████▊| 49/50 [9:38:11<11:17, 677.50s/it]

 mom=108.803 Gamma=108.807 beta=0.999958
 mom=105.021 Gamma=105.026 beta=0.999955
 mom=97.8503 Gamma=97.8554 beta=0.999948
 mom=88.0103 Gamma=88.016 beta=0.999935
 mom=76.4273 Gamma=76.4338 beta=0.999914
 mom=64.0915 Gamma=64.0993 beta=0.999878
 mom=51.9205 Gamma=51.9302 beta=0.999815
 mom=40.6536 Gamma=40.6659 beta=0.999698
 mom=30.793 Gamma=30.8093 beta=0.999473
 mom=22.5939 Gamma=22.6161 beta=0.999022
 mom=16.0939 Gamma=16.125 beta=0.998075
 mom=11.1671 Gamma=11.2118 beta=0.996014
 mom=7.58717 Gamma=7.65279 beta=0.991426
 mom=5.08472 Gamma=5.18212 beta=0.981204
 mom=3.39245 Gamma=3.53677 beta=0.959195
 mom=2.27465 Gamma=2.48476 beta=0.915441
 mom=1.5428 Gamma=1.83854 beta=0.839144
 mom=1.0596 Gamma=1.45697 beta=0.727264
 mom=0.733702 Gamma=1.24029 beta=0.591557
 mom=0.508632 Gamma=1.12192 beta=0.453358
 mom=0.350637 Gamma=1.05969 beta=0.330886
 mom=282.022 Gamma=282.023 beta=0.999994
 mom=221.272 Gamma=221.274 beta=0.99999
 mom=136.33 Gamma=136.333 beta=0.999973
 mom=66.1654 Gamma=6

[ ERROR   ] : [ blastwave.h:1527 ] : /tmp/ipykernel_1985683/1492004778.py:89: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current_row['theta_id'] = theta_id
/tmp/ipykernel_1985683/1492004778.py:89: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current_row['theta_id'] = theta_id
100%|██████████| 50/50 [9:51:24<00:00, 709.69s/it]

 mom=182.946 Gamma=182.949 beta=0.999985
 mom=150.513 Gamma=150.516 beta=0.999978
 mom=101.959 Gamma=101.964 beta=0.999952
 mom=57.0166 Gamma=57.0253 beta=0.999846
 mom=26.5282 Gamma=26.5471 beta=0.99929
 mom=10.5254 Gamma=10.5728 beta=0.995517
 mom=3.81887 Gamma=3.94763 beta=0.967383
 mom=1.43107 Gamma=1.74584 beta=0.819703
 mom=0.578113 Gamma=1.15508 beta=0.500495
 mom=0.231729 Gamma=1.0265 beta=0.225747
 mom=0.0863419 Gamma=1.00372 beta=0.0860219
 mom=0.0293042 Gamma=1.00043 beta=0.0292917
 mom=0.00902243 Gamma=1.00004 beta=0.00902206
 mom=0.0025184 Gamma=1 beta=0.00251839
 mom=0.000637234 Gamma=1 beta=0.000637233
 mom=0.000146165 Gamma=1 beta=0.000146165
 mom=3.03918e-05 Gamma=1 beta=3.03918e-05
  [RESAMPLED] theta_id=49 重试1次后成功

=== 生成完成 ===
成功样本数: 50 / 50
失败样本数: 0
重新采样后成功: 14

point_df.shape = (12800, 18)
fail_df.shape  = (empty)
resample_df.shape = (14, 3)


,theta_id,t_s,nu_hz,F_nu_mJy,z,theta_v,Gamma0,E_iso,theta_c,theta_w,n_ism,p,eps_e,eps_B,log10_t,log10_nu,valid,log10_F_nu_mJy
0,0,3000.000000,1.000000e+09,0.000008,0.133159,0.215264,282.178589,2.486071e+54,0.070959,0.410033,0.042564,2.167659,0.048042,0.000008,3.477121,9.0,True,-5.094097
1,0,3405.385129,1.000000e+09,0.000012,0.133159,0.215264,282.178589,2.486071e+54,0.070959,0.410033,0.042564,2.167659,0.048042,0.000008,3.532166,9.0,True,-4.928968
2,0,3865.549291,1.000000e+09,0.000017,0.133159,0.215264,282.178589,2.486071e+54,0.070959,0.410033,0.042564,2.167659,0.048042,0.000008,3.587211,9.0,True,-4.763846
3,0,4387.894690,1.000000e+09,0.000025,0.133159,0.215264,282.178589,2.486071e+54,0.070959,0.410033,0.042564,2.167659,0.048042,0.000008,3.642256,9.0,True,-4.598736
4,0,4980.823774,1.000000e+09,0.000037,0.133159,0.215264,282.178589,2.486071e+54,0.070959,0.410033,0.042564,2.167659,0.048042,0.000008,3.697301,9.0,True,-4.433669


## 八、保存数据

In [47]:

# 保存参数表（注意：这是经过重新采样后的最终参数）
theta_df.to_csv(OUT_THETA_CSV, index=False)
print(f'Saved: {OUT_THETA_CSV}  rows={len(theta_df)}')

# 保存点样本数据
try:
    point_df.to_parquet(OUT_PARQUET, index=False)
    print(f'Saved: {OUT_PARQUET}  rows={len(point_df)}')
except Exception as e:
    print('Parquet 保存失败（可能缺 pyarrow/fastparquet），改存 CSV。错误：', repr(e))
    point_df.to_csv(OUT_CSV, index=False)
    print(f'Saved: {OUT_CSV}  rows={len(point_df)}')

# 保存失败记录
if len(fail_df) > 0:
    fail_df.to_csv('pyblast_gaussian_failures_1000.csv', index=False)
    print('Saved: pyblast_gaussian_failures_1000.csv')

# 保存重新采样记录
if len(resample_df) > 0:
    resample_df.to_csv('pyblast_gaussian_resample_log_1000.csv', index=False)
    print('Saved: pyblast_gaussian_resample_log_1000.csv')


Saved: pyblast_gaussian_theta_table_1000.csv  rows=50
Parquet 保存失败（可能缺 pyarrow/fastparquet），改存 CSV。错误： ImportError("Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.\nA suitable version of pyarrow or fastparquet is required for parquet support.\nTrying to import the above resulted in these errors:\n - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.\n - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.")
Saved: pyblast_gaussian_point_dataset_1000.csv  rows=12800
Saved: pyblast_gaussian_resample_log_1000.csv


## 九、快速检查一组 light curve

In [48]:

example_theta_id = int(point_df['theta_id'].iloc[0])
sub = point_df[point_df['theta_id'] == example_theta_id].copy()

fig, ax = plt.subplots(figsize=(6, 4))
for freq, grp in sub.groupby('nu_hz'):
    grp = grp.sort_values('t_s')
    ax.plot(grp['t_s'] / cgs.day, grp['F_nu_mJy'], label=f'{freq:.2e} Hz')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('time [day]')
ax.set_ylabel('Flux density [mJy]')
ax.set_title(f'theta_id = {example_theta_id}')
ax.grid(ls=':')
ax.legend(fontsize=8)
plt.show()


RuntimeError: Failed to process string with tex because latex could not be found

<Figure size 600x400 with 1 Axes>

## 十、统计重新采样情况

In [49]:

if len(resample_df) > 0:
    print("=== 重新采样统计 ===")
    print(f"总共重新采样次数: {len(resample_df)}")
    print(f"重新采样后成功: {resample_df['final_success'].sum()}")
    print(f"重新采样后失败: {(~resample_df['final_success']).sum()}")
    print("\n重试次数分布:")
    print(resample_df['retries'].value_counts().sort_index())
else:
    print("所有样本首次运行即成功，无需重新采样！")


=== 重新采样统计 ===
总共重新采样次数: 14
重新采样后成功: 14
重新采样后失败: 0

重试次数分布:
retries
1    11
2     2
3     1
Name: count, dtype: int64
